# FinBERT Training on FinancialPhraseBank

This notebook extracts the FinancialPhraseBank dataset and fine-tunes a pre-trained model (like DistilRoBERTa) for financial sentiment classification.

In [7]:
# Install required libraries (use %pip for IPython/Jupyter compatibility)
# Note: We pin datasets<=2.21.0 because newer versions removed support for dataset scripts (like financial_phrasebank)
%pip install -q "datasets<=2.21.0" transformers torch scikit-learn accelerate evaluate

In [8]:
import os
import torch
import numpy as np
import evaluate # type: ignore
import warnings
warnings.filterwarnings('ignore')
from datasets import load_dataset # type: ignore
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

## 1. Data Preparation

Download the dataset from Hugging Face Hub (this avoids local file path issues on remote kernels) and prepare it.

In [9]:
# Load the allagree dataset from Hugging Face Hub
# We pass trust_remote_code=True as it is required for financial_phrasebank dataset loading
dataset = load_dataset('financial_phrasebank', 'sentences_allagree', trust_remote_code=True)

# Split data into train and validation sets
dataset = dataset['train'].train_test_split(test_size=0.1, stratify_by_column="label", seed=42) # type: ignore
train_dataset = dataset['train'] # type: ignore
val_dataset = dataset['test'] # type: ignore

print(f"Loaded {len(train_dataset)} training sentences and {len(val_dataset)} validation sentences.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'financial_phrasebank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'financial_phrasebank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found financial_phrasebank.py

## 2. Tokenization and Model Setup

In [ ]:
MODEL_NAME = 'distilroberta-base' # Lightweight model suitable for CPU/FastAPI inference

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True) # type: ignore
tokenized_val = val_dataset.map(tokenize_function, batched=True) # type: ignore

# Define label mappings so the API knows the string labels
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device) # type: ignore
print(f"Using device: {device}")

## 3. Training

In [ ]:
metric = evaluate.load("accuracy") # type: ignore

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels) # type: ignore

training_args = TrainingArguments( # type: ignore
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # type: ignore
    eval_dataset=tokenized_val, # type: ignore
    compute_metrics=compute_metrics
)

trainer.train()

## 4. Save the Model

In [ ]:
# If you are running this on Colab, you should mount your Google Drive 
# or download these files to your local machine afterwards.
SAVE_PATH = '../models/sentiment_model'
os.makedirs(SAVE_PATH, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(SAVE_PATH) # type: ignore
tokenizer.save_pretrained(SAVE_PATH) # type: ignore

print(f"Model and tokenizer successfully saved to {SAVE_PATH}")